# Whisper testing

This notebook is used to compare Whisper efficiency and accuracy based on the model used (small, base, tiny etc) and other parameters. During end-to-end testing of the application pipeline, latency is prohibitive, with Whisper being one of the components contributing the most to it with an average latency of 2.919 seconds per sentence passed.

The dataset used to determine the bast Whisper variant is espnet / yodas-granary from Hugging Face (https://huggingface.co/datasets/espnet/yodas-granary). This is a modified version of the larger nvidia/Granary dataset, specifically designed for ASR across 23 languages. The transcriptions of the audio clips have been derived faster-whisper-large-v3 model. Given the size and complexity of this model, most transcriptions are expected to be correct. However, in the future the accuracy of the dataset should be examined. Given that for now the application only supports English, the dataset is filtered to extract the entries in English. If we decide to extend to other languages later, this dataset will be helpful in testing Whisper in other languages as well.

## Loading

In [9]:
%pip install --upgrade pip
%pip install faster-whisper
%pip install pandas
%pip install numpy
%pip install jiwer
%pip install datasets
%pip install torch
%pip install -U datasets[audio]

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


If you face issues with torchcodec not being installed, even though the above code has beeen run, make sure ffpmeg is installed and run the following 4 cells:

In [10]:
%pip show torchcodec
!ffmpeg

Name: torchcodec
Version: 0.16.0
Summary: A video decoder for PyTorch
Home-page: 
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: 
Location: c:\Users\Administrator\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages
Requires: 
Required-by: 
Note: you may need to restart the kernel to use updated packages.


ffmpeg version 9.0.1-essentials_build-www.gyan.dev Copyright (c) 2000-2026 the FFmpeg developers
  built with gcc 16.1.0 (Rev2, Built by MSYS2 project)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-cairo --enable-fontconfig --enable-iconv --enable-gnutls --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-zlib --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-sdl2 --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxvid --enable-libaom --enable-libopenjpeg --enable-libvpx --enable-mediafoundation --enable-libass --enable-libfreetype --enable-libfribidi --enable-libharfbuzz --enable-libvidstab --enable-libvmaf --enable-libzimg --enable-amf --enable-cuda-llvm --enable-cuvid --enable-dxva2 --enable-d3d11va --enable-d3d12va --enable-ffnvcodec --enable-libvpl --enable-nvdec --enable-nvenc --enable-vaapi --enable-openal --enable-libgme --enable-libopenmpt --enable-libopenc

In [11]:
from datasets import load_dataset
import torch
from faster_whisper import WhisperModel
import pandas as pd
import jiwer
import time
import torchcodec
import numpy as np

In [12]:
ds = load_dataset("espnet/yodas-granary", "English", streaming = True)            # Load only english
print(ds.column_names)

{'asr_only': ['utt_id', 'audio', 'duration', 'lang', 'task', 'text', 'translation_en', 'original_audio_id', 'original_audio_offset']}


## Testing


To test the affect of changing the Whisper model variant in accuracy and performance the following models are used "small.en", "distil-small.en", "tiny.en", "base.en", "distil-medium.en", "large-v1", "distil-large-v2", "distil-large-v3", "large-v3-turbo", and "turbo".

On the one hand, distil-small, distil-large and distil-medium were trained by freezing the encoder layers and using only the first and the last decoder layers from the original whisper model, discarding the rest. According to the creators, distiled Whisper is 6 times faster, 49% smaller and performs within 1% WER of the original model. Moreover, it is more robust to noise and hallucinations. On the other hand, turbo whisper is inspired from distil whisper and it is an optimization of whipser-large-v3. It hs only 4 decoder layers instead of 32, but instead of using distillation, the model is trained for two more epochs over the same amount of data as large-v3. To evaluate the models above, 2 metrics are used:

* WER : calculates substitutions, deletions and insertions on word level measuring the accuracy of the model
* RTFx (Inverse real time factor) : ratio of duration divided by processing time measuring the latency of the model

### Assumption

The assumption used during testing is that the audio clips are shorter than 30 seconds, which is the maximum receptive field of the Whisper models. It is crucial that the possibility of larger audio clips provided by the user is considered. In this case, chunking is needed to process the audio clips. In the first round of testing with no quantization, the precision used is float16.

### Optimizations

In [13]:
# Picking GPU device if available
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
print(f"Device : {device} \n")

Device : cuda 



In [14]:
model_ids = ["small.en", "distil-small.en", "tiny.en", "base.en", "distil-medium.en", "large-v1", "distil-large-v2", "distil-large-v3", "large-v3-turbo", "turbo"]


In [15]:
# Transformations for WER
results = pd.DataFrame(columns = ['model', 'wer', 'rtf', 'time'])
for id in model_ids:
  print(f"Model : {id} \n")
  df = iter(ds['asr_only'])
  rtfList = np.zeros(100)
  werList = np.zeros(100)
  timeList = np.zeros(100)

  # Loading model
  model = WhisperModel(
      model_size_or_path = id,
      device = device,
      compute_type = "float16",               # Tuned parameter
      use_auth_token = True,
      # flash_attention = True,               # Tuned parameter
      # tensor_parallel = True,               # Tuned parameter
  )

  for index in range(100):
    # Inference
    item = next(df)
    start = time.perf_counter()
    segments, _ = model.transcribe(
        item["audio"]["array"],
        beam_size = 5,                          # Tuned parameter
        # vad_filter = True                     # Tuned parameter
        # condition_on_previous_text = False    # Tuned parameter
    )
    result = " ".join([segment.text for segment in segments])
    end = time.perf_counter()
    duration = end - start

    # World error rate
    reference = item["text"]
    wer = jiwer.wer(
      reference,
      result
    )

    # Inverse real time factor
    rtf = item["duration"] / duration

    # Storing results
    werList[index] = wer
    rtfList[index] = rtf
    timeList[index] = duration

  averageWER = np.mean(werList)
  averageRTF = np.mean(rtfList)
  averageTime = np.mean(timeList)

  print(f"WER for {id} : {averageWER} \n")
  print(f"RTF for {id} : {averageRTF} \n")
  print(f"Duration for {id} : {averageTime} \n")

  # Update pandas dataframe of results
  results = pd.concat([pd.DataFrame([[id, averageWER, averageRTF, averageTime]], columns=results.columns), results], ignore_index=True)
  print(results)



Model : small.en 



RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, 8, and 9, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.11.0+cu128) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 9:
Traceback (most recent 

## Parameter optimizatioon

After the optimal model has been found from the above with the optimization techniques outlined, several parameters of the model are tweaked to deduce whether they improve latency significantly. Those are:
1. disabled condition on previous text to reduce latency
2. enabled vad filter for reduced latency and address hallucination
3. beam size changed to 1 to reduce latency
4. flash attention enabled
5. tensor parallelism enabled
6. quantization to int8

The above are used on the base Whisper model.

In [ ]:
results = pd.DataFrame(columns = ['technique', 'wer', 'rtf', 'time'])

In [ ]:
# Quantization
df = iter(ds['asr_only'])
rtfList = np.zeros(100)
werList = np.zeros(100)
timeList = np.zeros(100)

model = WhisperModel(
    model_size_or_path = "base.en",
    device = device,
    compute_type = "int8",
    use_auth_token = True
)

for index in range(100):
    # Inference
    item = next(df)
    start = time.perf_counter()
    segments, _ = model.transcribe(
        item["audio"]["path"],
        beam_size = 5
    )
    result = " ".join([segment.text for segment in segments])
    end = time.perf_counter()
    duration = end - start

    # World error rate
    reference = item["text"]
    wer = jiwer.wer(
      reference,
      result
    )

    # Inverse real time factor
    rtf = item["duration"] / duration

    # Storing results
    werList[index] = wer
    rtfList[index] = rtf
    timeList[index] = duration

averageWER = np.mean(werList)
averageRTF = np.mean(rtfList)
averageTime = np.mean(timeList)

print(f"WER for quantization : {averageWER} \n")
print(f"RTF for quantization : {averageRTF} \n")
print(f"Duration for quantization : {averageTime} \n")

results = pd.concat([pd.DataFrame([['quantization', averageWER, averageRTF, averageTime]], columns=results.columns), results], ignore_index=True)


ImportError: To support decoding audio data, please install 'torchcodec'.

In [ ]:
# Flash attention and parallelism
df = iter(ds['asr_only'])
rtfList = np.zeros(100)
werList = np.zeros(100)
timeList = np.zeros(100)

model = WhisperModel(
    model_size_or_path = "base.en",
    device = device,
    compute_type = "float16",
    use_auth_token = True,
    flash_attention = True,
    tensor_parallel = True
)

for index in range(100):
    # Inference
    item = next(df)
    start = time.perf_counter()
    segments, _ = model.transcribe(
        item["audio"]["array"],
        beam_size = 5
    )
    result = " ".join([segment.text for segment in segments])
    end = time.perf_counter()
    duration = end - start

    # World error rate
    reference = item["text"]
    wer = jiwer.wer(
      reference,
      result
    )

    # Inverse real time factor
    rtf = item["duration"] / duration

    # Storing results
    werList[index] = wer
    rtfList[index] = rtf
    timeList[index] = duration

averageWER = np.mean(werList)
averageRTF = np.mean(rtfList)
averageTime = np.mean(timeList)

print(f"WER for flash attention : {averageWER} \n")
print(f"RTF for flash attention : {averageRTF} \n")
print(f"Duration for flash attention : {averageTime} \n")

results = pd.concat([pd.DataFrame([['flash', averageWER, averageRTF, averageTime]], columns=results.columns), results], ignore_index=True)


RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, 8, and 9, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.11.0+cu128) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 9:
Traceback (most recent 

In [ ]:
model = WhisperModel(
    model_size_or_path = "base.en",
    device = device,
    compute_type = "float16",
    use_auth_token = True,
)

# Beam size
for index in range(100):
    # Inference
    item = next(df)
    start = time.perf_counter()
    segments, _ = model.transcribe(
        item["audio"]["array"],
        beam_size = 1
    )
    result = " ".join([segment.text for segment in segments])
    end = time.perf_counter()
    duration = end - start

    # World error rate
    reference = item["text"]
    wer = jiwer.wer(
      reference,
      result
    )

    # Inverse real time factor
    rtf = item["duration"] / duration

    # Storing results
    werList[index] = wer
    rtfList[index] = rtf
    timeList[index] = duration

averageWER = np.mean(werList)
averageRTF = np.mean(rtfList)
averageTime = np.mean(timeList)

print(f"WER for beam size : {averageWER} \n")
print(f"RTF for beam size : {averageRTF} \n")
print(f"Duration for beam size : {averageTime} \n")

results = pd.concat([pd.DataFrame([['beam', averageWER, averageRTF, averageTime]], columns=results.columns), results], ignore_index=True)

# Beam filter
for index in range(100):
    # Inference
    item = next(df)
    start = time.perf_counter()
    segments, _ = model.transcribe(
        item["audio"]["array"],
        beam_size = 5,
        vad_filter = True
    )
    result = " ".join([segment.text for segment in segments])
    end = time.perf_counter()
    duration = end - start

    # World error rate
    reference = item["text"]
    wer = jiwer.wer(
      reference,
      result
    )

    # Inverse real time factor
    rtf = item["duration"] / duration

    # Storing results
    werList[index] = wer
    rtfList[index] = rtf
    timeList[index] = duration

averageWER = np.mean(werList)
averageRTF = np.mean(rtfList)
averageTime = np.mean(timeList)

print(f"WER for vad : {averageWER} \n")
print(f"RTF for vad : {averageRTF} \n")
print(f"Duration for vad : {averageTime} \n")

results = pd.concat([pd.DataFrame([['vad', averageWER, averageRTF, averageTime]], columns=results.columns), results], ignore_index=True)

# Previous text
for index in range(100):
    # Inference
    item = next(df)
    start = time.perf_counter()
    segments, _ = model.transcribe(
        item["audio"]["array"],
        beam_size = 5,
        condition_on_previous_text = False
    )
    result = " ".join([segment.text for segment in segments])
    end = time.perf_counter()
    duration = end - start

    # World error rate
    reference = item["text"]
    wer = jiwer.wer(
      reference,
      result
    )

    # Inverse real time factor
    rtf = item["duration"] / duration

    # Storing results
    werList[index] = wer
    rtfList[index] = rtf
    timeList[index] = duration

averageWER = np.mean(werList)
averageRTF = np.mean(rtfList)
averageTime = np.mean(timeList)

print(f"WER for previous text : {averageWER} \n")
print(f"RTF for previous text : {averageRTF} \n")
print(f"Duration for previous text : {averageTime} \n")

results = pd.concat([pd.DataFrame([['previous', averageWER, averageRTF, averageTime]], columns=results.columns), results], ignore_index=True)

# Benchamrk
for index in range(100):
    # Inference
    item = next(df)
    start = time.perf_counter()
    segments, _ = model.transcribe(
        item["audio"]["array"],
        beam_size = 5
    )
    result = " ".join([segment.text for segment in segments])
    end = time.perf_counter()
    duration = end - start

    # World error rate
    reference = item["text"]
    wer = jiwer.wer(
      reference,
      result
    )

    # Inverse real time factor
    rtf = item["duration"] / duration

    # Storing results
    werList[index] = wer
    rtfList[index] = rtf
    timeList[index] = duration

averageWER = np.mean(werList)
averageRTF = np.mean(rtfList)
averageTime = np.mean(timeList)

print(f"WER (benchmark) : {averageWER} \n")
print(f"RTF (benchmark) : {averageRTF} \n")
print(f"Duration (benchmark) : {averageTime} \n")

results = pd.concat([pd.DataFrame([['Plain', averageWER, averageRTF, averageTime]], columns=results.columns), results], ignore_index=True)

ValueError: Requested float16 compute type, but the target device or backend do not support efficient float16 computation.